In [1]:
!pip install "opencv-python<5" "opencv-contrib-python<5" -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install "numpy==1.23.5" -i https://pypi.tuna.tsinghua.edu.cn/simple
import numpy
import cv2
from d2l import torch as d2l
print("numpy:", numpy.__version__)
print("opencv:", cv2.__version__)
print("d2l 导入成功！")

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 MB 65.0 MB/s  0:00:01 67.1 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 MB 76.6 MB/s  0:00:01 80.6 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 74.1 MB/s  0:00:009.5 MB/s eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: opencv-python━━━━━━━━━━━━━━━━━ 0/3 [numpy]
    Found existing installation: opencv-python 5.0.0.930/3 [numpy]
    Uninstalling opencv-python-5.0.0.93:━━━━━━━━━━ 0/3 [numpy]
      Successfully uninstalled opencv-python-5.0.0.93m 0/3 [numpy]
  Attempting uninstall: opencv-contrib-python237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [opencv-python]
    Found existing installation: opencv-contrib-python 5.0.0.93━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [opencv-python]
    Uninstalling open

In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [3]:
#残差块
#定义残差块
class Residual(nn.Module):
#初始化
    def __init__(self,input_channels,num_channels,use_1x1conv=False,strides=1):
        super().__init__()
#定义第一个卷积层
        self.conv1 = nn.Conv2d(input_channels,num_channels,kernel_size=3,padding=1,stride=strides)
#定义第二个卷积层
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3,padding=1)
#定义第三个卷积层
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels,num_channels,kernel_size=1,stride=strides)
        else:
            self.conv3 = None
#定义批量归一化层
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)
#定义激活函数
        self.relu = nn.ReLU(inplace=True)
#定义向前传播的函数
    def forward(self,X):
        Y =F.relu(self.bn1(self.conv1(X)))
        Y =self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

In [4]:
#如果输入和输出形状一致
blk = Residual(3,3)
X = torch.rand(4,3,6,6)
Y = blk(X)
Y.shape

torch.Size([4, 3, 6, 6])

In [5]:
#增加输出通道数的同时，减半输出的高和宽
blk = Residual(3,6,use_1x1conv=True,strides=2)
blk(X).shape

torch.Size([4, 6, 3, 3])

In [ ]:
#定义Resnet模型
#b1:初始卷积层
b1 = nn.Sequential(nn.Conv2d(1,64,kernel_size=7,stride=2,padding=3)
                   nn.BatchNorm2d(64),nn.ReLU()<
                   nn.MaxPool2d(kernel_size=3,stride=2,padding=1))
#生成第二个stage
def resnet_block(input_channels,num_channels,num_residuals,first_block=False):
#创建空列表，循环生成残差块
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels,num_channels,
    